# 01 — Data Collection and Integration

Collect hourly OpenAQ pollutants and Open-Meteo weather for Hanoi (2025-08-01 to 2026-08-01, UTC). The station is selected using **one-year PM2.5 completeness**; data sources are then combined with **full outer joins**.

## 1. Configuration

In [36]:
import json
from getpass import getpass
from pathlib import Path

import pandas as pd
import requests

BASE_URL = "https://api.openaq.org/v3"
API_KEY = getpass("Enter your OpenAQ API key: ").strip()
if not API_KEY:
    raise ValueError("OpenAQ API key cannot be empty.")
HEADERS = {"X-API-Key": API_KEY}

STATIONS = {
    2161292: "Luu_Quang_Vu",
    2161306: "Minh_Khai",
    4946813: "Bach_Khoa",
}
SAMPLE_START = "2026-08-01T00:00:00Z"
SAMPLE_END = "2026-08-08T00:00:00Z"
YEAR_SAMPLE_START = "2025-08-01T00:00:00Z"
YEAR_SAMPLE_END = "2026-08-01T00:00:00Z"

## 2. Station metadata and sensor inventory

In [37]:
def get_location(location_id):
    url = f"{BASE_URL}/locations/{location_id}"

    response = requests.get(
        url,
        headers=HEADERS,
        timeout=30
    )

    response.raise_for_status()

    return response.json()

def get_sensors(location_id):

    url = f"{BASE_URL}/locations/{location_id}/sensors"

    response = requests.get(
        url,
        headers=HEADERS,
        timeout=30
    )

    response.raise_for_status()

    return response.json()

In [38]:
location_metadata = {}
all_sensor_data = {}
RAW_DIR = Path("../data/raw/openaq/metadata")
RAW_DIR.mkdir(parents=True, exist_ok=True)

for location_id, station_name in STATIONS.items():
    location_metadata[location_id] = get_location(location_id)
    all_sensor_data[location_id] = get_sensors(location_id)
    with open(RAW_DIR / f"location_{location_id}.json", "w", encoding="utf-8") as f:
        json.dump(location_metadata[location_id], f, ensure_ascii=False, indent=2)
    with open(RAW_DIR / f"sensors_{location_id}.json", "w", encoding="utf-8") as f:
        json.dump(all_sensor_data[location_id], f, ensure_ascii=False, indent=2)

records = []

for location_id, station_name in STATIONS.items():

    sensors = all_sensor_data[location_id]["results"]

    for sensor in sensors:

        records.append({
            "location_id": location_id,
            "station": station_name,

            "sensor_id": sensor["id"],

            "parameter": sensor["parameter"]["name"],
            "unit": sensor["parameter"]["units"],

            "first_utc": (
                sensor["datetimeFirst"]["utc"]
                if sensor["datetimeFirst"]
                else None
            ),

            "last_utc": (
                sensor["datetimeLast"]["utc"]
                if sensor["datetimeLast"]
                else None
            ),

            "raw_observed_count": (
                sensor["coverage"]["observedCount"]
                if sensor["coverage"]
                else None
            )
        })

sensor_inventory = pd.DataFrame(records)

sensor_inventory

# Sensors relevant to the study
sensor_inventory[sensor_inventory["parameter"].isin(["pm25", "pm10", "no2"])]

,location_id,station,sensor_id,parameter,unit,first_utc,last_utc,raw_observed_count
3,2161292,Luu_Quang_Vu,7772032,pm25,µg/m³,2024-01-29T16:00:00Z,2026-10-14T02:00:00Z,17282
4,2161292,Luu_Quang_Vu,7772048,pm10,µg/m³,2024-01-29T16:00:00Z,2026-10-14T02:00:00Z,10836
5,2161292,Luu_Quang_Vu,7888636,no2,µg/m³,2024-02-19T03:00:00Z,2026-07-13T17:00:00Z,11661
8,2161306,Minh_Khai,7772055,pm10,µg/m³,2024-01-29T16:00:00Z,2026-06-03T01:00:00Z,9251
9,2161306,Minh_Khai,7772087,pm25,µg/m³,2024-01-29T16:00:00Z,2026-10-14T02:00:00Z,7739
11,2161306,Minh_Khai,7772100,no2,µg/m³,2024-01-29T16:00:00Z,2026-06-03T01:00:00Z,2556
13,4946813,Bach_Khoa,13502155,no2,µg/m³,2025-07-03T15:40:00Z,2026-10-02T09:05:00Z,81636
14,4946813,Bach_Khoa,13502154,pm25,µg/m³,2025-07-03T15:40:00Z,2026-10-14T09:35:00Z,110487
15,4946813,Bach_Khoa,13502152,pm10,µg/m³,2025-07-03T15:40:00Z,2026-10-14T09:35:00Z,114682


## 3. PM2.5 availability and station selection

In [39]:
def get_hourly_measurements(
    sensor_id,
    datetime_from,
    datetime_to,
    page=1,
    limit=1000
):
    url = f"{BASE_URL}/sensors/{sensor_id}/hours"

    params = {
        "datetime_from": datetime_from,
        "datetime_to": datetime_to,
        "page": page,
        "limit": limit
    }

    response = requests.get(
        url,
        headers=HEADERS,
        params=params,
        timeout=30
    )

    response.raise_for_status()

    return response.json()

def parse_hourly_measurements(response_json):
    records = []

    for row in response_json["results"]:
        records.append({
            "timestamp_utc":
                row["period"]["datetimeFrom"]["utc"],

            "timestamp_local":
                row["period"]["datetimeFrom"]["local"],

            "value":
                row["value"],

            "parameter":
                row["parameter"]["name"],

            "unit":
                row["parameter"]["units"],

            "has_flags":
                row["flagInfo"]["hasFlags"],

            "expected_count":
                row["coverage"]["expectedCount"],

            "observed_count":
                row["coverage"]["observedCount"],

            "percent_complete":
                row["coverage"]["percentComplete"],

            "percent_coverage":
                row["coverage"]["percentCoverage"]
        })

    return pd.DataFrame(records)

def get_all_hourly_measurements(
    sensor_id,
    datetime_from,
    datetime_to,
    limit=1000
):
    all_results = []
    page = 1

    while True:
        response = get_hourly_measurements(
            sensor_id=sensor_id,
            datetime_from=datetime_from,
            datetime_to=datetime_to,
            page=page,
            limit=limit
        )

        results = response["results"]

        all_results.extend(results)

        print(
            f"Page {page}: "
            f"{len(results)} records "
            f"| total collected: {len(all_results)}"
        )

        # No data returned
        if len(results) == 0:
            break

        # Last page (less than 1000 a page)
        if len(results) < limit:
            break

        page += 1

    return {
        "results": all_results
    }

In [40]:
pm25_sensors = (
    sensor_inventory[sensor_inventory["parameter"] == "pm25"][["location_id", "station", "sensor_id"]]
    .reset_index(drop=True)
)

pm25_sensors

sample_data = {}
sample_raw_responses = {}

for _, row in pm25_sensors.iterrows():
    station_name = row["station"]
    sensor_id = row["sensor_id"]

    print(f"Collecting {station_name}...")

    response = get_hourly_measurements(
        sensor_id=sensor_id,
        datetime_from=SAMPLE_START,
        datetime_to=SAMPLE_END
    )

    df = parse_hourly_measurements(response)

    df["timestamp_utc"] = pd.to_datetime(
        df["timestamp_utc"],
        utc=True
    )

    df["timestamp_local"] = pd.to_datetime(
        df["timestamp_local"]
    )

    df["station"] = station_name
    df["sensor_id"] = sensor_id

    sample_raw_responses[station_name] = response
    sample_data[station_name] = df

    print(
        station_name,
        "records:",
        len(df)
    )

expected_hours = pd.date_range(
    start=SAMPLE_START,
    end=SAMPLE_END,
    freq="h",
    inclusive="left"
)

audit_records = []

for station_name, df in sample_data.items():

    observed_hours = pd.DatetimeIndex(
        df["timestamp_utc"].unique()
    )

    observed_in_window = expected_hours.intersection(
        observed_hours
    )

    missing_hours = expected_hours.difference(
        observed_hours
    )

    audit_records.append({
        "station": station_name,
        "expected_hours": len(expected_hours),
        "observed_hours": len(observed_in_window),
        "missing_hours": len(missing_hours),
        "completeness_pct":
            len(observed_in_window)
            / len(expected_hours)
            * 100
    })

sample_audit = pd.DataFrame(audit_records)

sample_audit

Luu_Quang_Vu records: 61
Minh_Khai records: 73
Bach_Khoa records: 161


,station,expected_hours,observed_hours,missing_hours,completeness_pct
0,Luu_Quang_Vu,168,61,107,36.309524
1,Minh_Khai,168,73,95,43.452381
2,Bach_Khoa,168,161,7,95.833333


In [41]:
sample_data = {}
sample_raw_responses = {}

for _, row in pm25_sensors.iterrows():
    station_name = row["station"]
    sensor_id = row["sensor_id"]

    print(f"Collecting {station_name}...")

    response = get_all_hourly_measurements(
        sensor_id=sensor_id,
        datetime_from=YEAR_SAMPLE_START,
        datetime_to=YEAR_SAMPLE_END
    )

    df = parse_hourly_measurements(response)

    df["timestamp_utc"] = pd.to_datetime(
        df["timestamp_utc"],
        utc=True
    )

    df["timestamp_local"] = pd.to_datetime(
        df["timestamp_local"]
    )

    df["station"] = station_name
    df["sensor_id"] = sensor_id

    sample_raw_responses[station_name] = response
    sample_data[station_name] = df

    print(
        station_name,
        "records:",
        len(df)
    )

expected_hours = pd.date_range(
    start=YEAR_SAMPLE_START,
    end=YEAR_SAMPLE_END,
    freq="h",
    inclusive="left"
)

audit_records = []

for station_name, df in sample_data.items():

    observed_hours = pd.DatetimeIndex(
        df["timestamp_utc"].unique()
    )

    observed_in_window = expected_hours.intersection(
        observed_hours
    )

    missing_hours = expected_hours.difference(
        observed_hours
    )

    audit_records.append({
        "station": station_name,
        "expected_hours": len(expected_hours),
        "observed_hours": len(observed_in_window),
        "missing_hours": len(missing_hours),
        "completeness_pct":
            len(observed_in_window)
            / len(expected_hours)
            * 100
    })

sample_audit = pd.DataFrame(audit_records)

sample_audit

Page 1: 1000 records | total collected: 1000
Page 2: 1000 records | total collected: 2000
Page 3: 1000 records | total collected: 3000
Page 4: 1000 records | total collected: 4000
Page 5: 1000 records | total collected: 5000
Page 6: 37 records | total collected: 5037
Luu_Quang_Vu records: 5037
Page 1: 1000 records | total collected: 1000
Page 2: 263 records | total collected: 1263
Minh_Khai records: 1263
Page 1: 1000 records | total collected: 1000
Page 2: 1000 records | total collected: 2000
Page 3: 1000 records | total collected: 3000
Page 4: 1000 records | total collected: 4000
Page 5: 1000 records | total collected: 5000
Page 6: 1000 records | total collected: 6000
Page 7: 1000 records | total collected: 7000
Page 8: 1000 records | total collected: 8000
Page 9: 82 records | total collected: 8082
Bach_Khoa records: 8082


,station,expected_hours,observed_hours,missing_hours,completeness_pct
0,Luu_Quang_Vu,8760,5037,3723,57.500000
1,Minh_Khai,8760,1263,7497,14.417808
2,Bach_Khoa,8760,8082,678,92.260274


In [42]:
df_bk = sample_data["Bach_Khoa"].copy()
df_bk = df_bk.sort_values("timestamp_utc").reset_index(drop=True)
df_bk["time_diff"] = df_bk["timestamp_utc"].diff()

gaps_bk = df_bk[df_bk["time_diff"] > pd.Timedelta(hours=1)][["timestamp_utc", "time_diff"]]
print("Bach Khoa PM2.5 gaps > 1 hour:", len(gaps_bk))
print("Maximum interval between consecutive observations:", df_bk["time_diff"].max())

Bach Khoa PM2.5 gaps > 1 hour: 97
Maximum interval between consecutive observations: 5 days 17:00:00


/var/folders/sb/pmlt6jxs48zgnsjrc491m_s80000gn/T/ipykernel_91020/3674615154.py:5: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`).Please use a specific unit instead.
  gaps_bk = df_bk[df_bk["time_diff"] > pd.Timedelta(hours=1)][["timestamp_utc", "time_diff"]]


In [43]:
# Create a complete hourly timeline for the 1 year audit window
expected_hours = pd.date_range(
    start=YEAR_SAMPLE_START,
    end=YEAR_SAMPLE_END,
    freq="h",
    inclusive="left"
)

# Create expected timeline into a DataFrame
expected_df = pd.DataFrame({
    "timestamp_utc":expected_hours
})

# Create month label
expected_df["month"] = expected_df["timestamp_utc"].dt.to_period("M")

monthly_expected = (
    expected_df
    .groupby("month")
    .size()
    .rename("expected_hours")
)

observed_df = df_bk[["timestamp_utc"]].copy()

observed_df["month"] = (
    observed_df["timestamp_utc"].dt.to_period("M")
)

monthly_observed = (
    observed_df
    .groupby("month")
    ["timestamp_utc"]
    .nunique()
    .rename("observed_hours")
)

monthly_completeness = pd.concat(
    [
        monthly_expected,
        monthly_observed
    ],
    axis=1
)

monthly_completeness["observed_hours"] = (
    monthly_completeness["observed_hours"]
    .fillna(0)
    .astype(int)
)

monthly_completeness["missing_hours"] = (
    monthly_completeness["expected_hours"]
    - monthly_completeness["observed_hours"]
)

monthly_completeness["completeness_pct"] = (
    monthly_completeness["observed_hours"]
    / monthly_completeness["expected_hours"]
    * 100
).round(2)

monthly_completeness.sort_values(
    "completeness_pct"
)

print("Total missing hours:", monthly_completeness["missing_hours"].sum())

Total missing hours: 678


/var/folders/sb/pmlt6jxs48zgnsjrc491m_s80000gn/T/ipykernel_91020/973598381.py:15: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  expected_df["month"] = expected_df["timestamp_utc"].dt.to_period("M")
/var/folders/sb/pmlt6jxs48zgnsjrc491m_s80000gn/T/ipykernel_91020/973598381.py:27: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  observed_df["timestamp_utc"].dt.to_period("M")


**Station selected:** Bach Khoa, based on the full-year completeness comparison above.

## 4. PM10 and NO2 at Bach Khoa

In [44]:
pm10_sensors = (
    sensor_inventory[
        (sensor_inventory["parameter"] == "pm10")
        & (sensor_inventory["station"] == "Bach_Khoa") # (condition_1) & (condition_2)
    ][
        ["location_id", "station", "sensor_id"]
    ]
    .reset_index(drop=True)
)

pm10_sensors

no2_sensors = (
    sensor_inventory[
        (sensor_inventory["parameter"] == "no2")
        & (sensor_inventory["station"] == "Bach_Khoa")
    ][
        ["location_id", "station", "sensor_id"]
    ]
    .reset_index(drop=True)
)

no2_sensors

response_pm10 = get_all_hourly_measurements(
    pm10_sensors.loc[0, "sensor_id"], 
    YEAR_SAMPLE_START, 
    YEAR_SAMPLE_END
)

df_pm10 = parse_hourly_measurements(response_pm10)
df_pm10["timestamp_utc"] = pd.to_datetime(
        df_pm10["timestamp_utc"],
        utc=True
    )

df_pm10["timestamp_local"] = pd.to_datetime(
    df_pm10["timestamp_local"]
    )

df_pm10["station"] = "Bach_Khoa"
df_pm10["sensor_id"] = pm10_sensors.loc[0, "sensor_id"]

print(
    "Duplicate timestamps:",
    df_pm10["timestamp_utc"].duplicated().sum()
)

df_pm10 = (
    df_pm10
    .sort_values("timestamp_utc")
    .reset_index(drop=True)
)

df_pm10["time_diff"] = df_pm10["timestamp_utc"].diff()

print(
    "Number of gaps > 1 hour:",
    (
        df_pm10["time_diff"]
        > pd.Timedelta(hours=1)
    ).sum()
)

print(
    "Maximum gap:",
    df_pm10["time_diff"].max()
)

Page 1: 1000 records | total collected: 1000
Page 2: 1000 records | total collected: 2000
Page 3: 1000 records | total collected: 3000
Page 4: 1000 records | total collected: 4000
Page 5: 1000 records | total collected: 5000
Page 6: 1000 records | total collected: 6000
Page 7: 1000 records | total collected: 7000
Page 8: 1000 records | total collected: 8000
Page 9: 169 records | total collected: 8169
Duplicate timestamps: 0
Number of gaps > 1 hour: 47
Maximum gap: 5 days 17:00:00


/var/folders/sb/pmlt6jxs48zgnsjrc491m_s80000gn/T/ipykernel_91020/496648554.py:61: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`).Please use a specific unit instead.
  > pd.Timedelta(hours=1)


In [45]:
response_no2 = get_all_hourly_measurements(
    no2_sensors.loc[0, "sensor_id"],
    YEAR_SAMPLE_START,
    YEAR_SAMPLE_END
)

df_no2 = parse_hourly_measurements(response_no2)

df_no2["timestamp_utc"] = pd.to_datetime(
        df_no2["timestamp_utc"],
        utc=True
    )

df_no2["timestamp_local"] = pd.to_datetime(
    df_no2["timestamp_local"]
    )

df_no2["station"] = "Bach_Khoa"
df_no2["sensor_id"] = no2_sensors.loc[0, "sensor_id"]

print(
    "Duplicate timestamps:",
    df_no2["timestamp_utc"].duplicated().sum()
)

df_no2 = (
    df_no2
    .sort_values("timestamp_utc")
    .reset_index(drop=True)
)

df_no2["time_diff"] = df_no2["timestamp_utc"].diff()

print(
    "Number of gaps > 1 hour:",
    (
        df_no2["time_diff"]
        > pd.Timedelta(hours=1)
    ).sum()
)

print(
    "Maximum gap:",
    df_no2["time_diff"].max()
)

Page 1: 1000 records | total collected: 1000
Page 2: 1000 records | total collected: 2000
Page 3: 1000 records | total collected: 3000
Page 4: 1000 records | total collected: 4000
Page 5: 1000 records | total collected: 5000
Page 6: 1000 records | total collected: 6000
Page 7: 1000 records | total collected: 7000
Page 8: 509 records | total collected: 7509
Duplicate timestamps: 0
Number of gaps > 1 hour: 30
Maximum gap: 19 days 11:00:00


/var/folders/sb/pmlt6jxs48zgnsjrc491m_s80000gn/T/ipykernel_91020/278666100.py:38: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`).Please use a specific unit instead.
  > pd.Timedelta(hours=1)


In [46]:
expected_hours = pd.date_range(
    start=YEAR_SAMPLE_START,
    end=YEAR_SAMPLE_END,
    freq="h",
    inclusive="left"
)

air_quality_data = {
    "pm25": df_bk,
    "pm10": df_pm10,
    "no2": df_no2
}

coverage_records = []

for parameter, df in air_quality_data.items():

    observed_hours = pd.DatetimeIndex(
        df["timestamp_utc"].unique()
    )

    observed_in_window = expected_hours.intersection(
        observed_hours
    )

    missing_hours = expected_hours.difference(
        observed_hours
    )

    completeness_pct = (
        len(observed_in_window)
        / len(expected_hours)
        * 100
    )

    coverage_records.append({
        "parameter": parameter,
        "expected_hours": len(expected_hours),
        "observed_hours": len(observed_in_window),
        "missing_hours": len(missing_hours),
        "completeness_pct": round(completeness_pct, 2)
    })

air_quality_coverage = pd.DataFrame(
    coverage_records
)

air_quality_coverage

,parameter,expected_hours,observed_hours,missing_hours,completeness_pct
0,pm25,8760,8082,678,92.26
1,pm10,8760,8169,591,93.25
2,no2,8760,7509,1251,85.72


## 5. Historical meteorological data

In [47]:
bach_khoa_metadata = location_metadata[4946813]["results"][0]

latitude = bach_khoa_metadata["coordinates"]["latitude"]
longitude = bach_khoa_metadata["coordinates"]["longitude"]

print("Latitude:", latitude)
print("Longitude:", longitude)

WEATHER_VARIABLES = [
    "temperature_2m",
    "relative_humidity_2m",
    "precipitation",
    "surface_pressure",
    "wind_speed_10m",
    "wind_direction_10m"
]

WEATHER_START_DATE = "2025-08-01"
WEATHER_END_DATE = "2026-07-31" # Till the end of the day

OPEN_METEO_URL = "https://archive-api.open-meteo.com/v1/archive"


def get_historical_weather(
    latitude,
    longitude,
    start_date,
    end_date,
    hourly_variables
):
    params = {
        "latitude": latitude,
        "longitude": longitude,
        "start_date": start_date,
        "end_date": end_date,
        "hourly": ",".join(hourly_variables),
        "timezone": "UTC"
    }

    response = requests.get(
        OPEN_METEO_URL,
        params=params,
        timeout=30
    )

    response.raise_for_status()

    return response.json()

Latitude: 21.0052
Longitude: 105.8418


In [48]:
weather_response = get_historical_weather(
    latitude=latitude,
    longitude=longitude,
    start_date=WEATHER_START_DATE,
    end_date=WEATHER_END_DATE,
    hourly_variables=WEATHER_VARIABLES
)

df_weather = pd.DataFrame(
    weather_response["hourly"]
)

df_weather = df_weather.rename(
    columns={
        "time": "timestamp_utc"
    }
)

df_weather["timestamp_utc"] = pd.to_datetime(
    df_weather["timestamp_utc"],
    utc=True
)

print("Rows:", len(df_weather))
print(
    "Duplicate timestamps:",
    df_weather["timestamp_utc"].duplicated().sum()
)

print(
    "Start:", 
    df_weather["timestamp_utc"].min()
)

print(
    "End:",
    df_weather["timestamp_utc"].max()
)

df_weather.isna().sum()

Rows: 8760
Duplicate timestamps: 0
Start: 2025-08-01 00:00:00+00:00
End: 2026-07-31 23:00:00+00:00


timestamp_utc           0
temperature_2m          0
relative_humidity_2m    0
precipitation           0
surface_pressure        0
wind_speed_10m          0
wind_direction_10m      0
dtype: int64

## 6. Integrate sources using full outer joins

In [49]:
pm25_hours = set(df_bk["timestamp_utc"])
pm10_hours = set(df_pm10["timestamp_utc"])
no2_hours = set(df_no2["timestamp_utc"])
weather_hours = set(df_weather["timestamp_utc"])

joint_hours = (
    pm25_hours
    & pm10_hours
    & no2_hours
    & weather_hours
)

print("Joint usable hours:", len(joint_hours))
print("Expected hours:", len(expected_hours))
print(
    "Joint completeness:",
    round(len(joint_hours) / len(expected_hours) * 100, 2),
    "%"
)

df_pm25_model = (
    df_bk[["timestamp_utc", "value"]]
    .rename(columns={"value": "pm25"})
)

df_pm10_model = (
    df_pm10[["timestamp_utc", "value"]]
    .rename(columns={"value": "pm10"})
)

df_no2_model = (
    df_no2[["timestamp_utc", "value"]]
    .rename(columns={"value": "no2"})
)

df_air_quality = (
    df_pm25_model
    .merge(
        df_pm10_model,
        on="timestamp_utc",
        how="outer"
    )
    .merge(
        df_no2_model,
        on="timestamp_utc",
        how="outer"
    )
)

df_merged = df_air_quality.merge(
    df_weather,
    on="timestamp_utc",
    how="outer"
)

df_merged = (
    df_merged
    .sort_values("timestamp_utc")
    .reset_index(drop=True)
)

Joint usable hours: 7231
Expected hours: 8760
Joint completeness: 82.55 %


## 7. Validate and save the hourly dataset

In [50]:
print("Shape:", df_merged.shape)

print(
    "Duplicate timestamps:",
    df_merged["timestamp_utc"].duplicated().sum()
)

df_merged.isna().sum()

INTERIM_DIR = Path("../data/interim")
INTERIM_DIR.mkdir(parents=True, exist_ok=True)

df_merged.to_pickle(
    INTERIM_DIR / "hanoi_air_quality_weather_hourly.pkl",
)

print("Saved: ../data/interim/hanoi_air_quality_weather_hourly.pkl")

Shape: (8760, 10)
Duplicate timestamps: 0
Saved: ../data/interim/hanoi_air_quality_weather_hourly.pkl
